In [ ]:
# Imports and Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, Subset
from pathlib import Path
from PIL import Image
import random
import time

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Kaggle Dataset Setup: Recyclable and Household Waste (30 classes)

This section keeps your MNIST experiments intact and adds a separate pipeline for Kaggle images.

Expected folder structure after unzip:

- `data/recyclable-household-waste/images/<category>/default/*.jpg`
- `data/recyclable-household-waste/images/<category>/real_world/*.jpg`

In [ ]:
# Configure dataset path and validate categories
from pathlib import Path

BASE_ROOT = Path("data/recyclable-household-waste/images")  # update if your folder is different

# Some archives unpack as images/images/<class>/...
candidate_roots = [BASE_ROOT / "images", BASE_ROOT]
DATA_ROOT = next((p for p in candidate_roots if p.exists()), None)

if DATA_ROOT is None:
    raise FileNotFoundError(
        f"Dataset folder not found under: {BASE_ROOT.resolve()}\n"
        "Download/unzip from Kaggle and update BASE_ROOT."
    )

# Category folders are labels
class_names = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])
num_classes = len(class_names)
print(f"Using DATA_ROOT: {DATA_ROOT.resolve()}")
print(f"Detected classes: {num_classes}")
print(class_names)

if num_classes != 30:
    print("Warning: Expected 30 categories for this dataset.")

Using DATA_ROOT: /home/teoyongsong/cnn/data/recyclable-household-waste/images/images
Detected classes: 30
['aerosol_cans', 'aluminum_food_cans', 'aluminum_soda_cans', 'cardboard_boxes', 'cardboard_packaging', 'clothing', 'coffee_grounds', 'disposable_plastic_cutlery', 'eggshells', 'food_waste', 'glass_beverage_bottles', 'glass_cosmetic_containers', 'glass_food_jars', 'magazines', 'newspaper', 'office_paper', 'paper_cups', 'plastic_cup_lids', 'plastic_detergent_bottles', 'plastic_food_containers', 'plastic_shopping_bags', 'plastic_soda_bottles', 'plastic_straws', 'plastic_trash_bags', 'plastic_water_bottles', 'shoes', 'steel_food_cans', 'styrofoam_cups', 'styrofoam_food_containers', 'tea_bags']


In [ ]:
# Build image samples from <class>/<default|real_world>/*.jpg|*.jpeg|*.png
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

class WasteImageDataset(Dataset):
    def __init__(self, root_dir: Path, class_names, transform=None):
        self.root_dir = root_dir
        self.class_to_idx = {name: idx for idx, name in enumerate(class_names)}
        self.transform = transform

        exts = {".jpg", ".jpeg", ".png", ".webp"}
        self.samples = []

        for cls in class_names:
            cls_path = self.root_dir / cls
            for sub in ["default", "real_world"]:
                sub_path = cls_path / sub
                if not sub_path.exists():
                    continue
                for img_path in sub_path.rglob("*"):
                    if img_path.suffix.lower() in exts:
                        self.samples.append((img_path, self.class_to_idx[cls]))

        if len(self.samples) == 0:
            raise RuntimeError("No images found. Check DATA_ROOT and folder structure.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# Create train/val/test splits and DataLoaders
import random
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = WasteImageDataset(DATA_ROOT, class_names, transform=None)

n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

indices = list(range(n_total))
rng = random.Random(42)
rng.shuffle(indices)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_dataset = Subset(WasteImageDataset(DATA_ROOT, class_names, transform=train_transform), train_idx)
val_dataset = Subset(WasteImageDataset(DATA_ROOT, class_names, transform=eval_transform), val_idx)
test_dataset = Subset(WasteImageDataset(DATA_ROOT, class_names, transform=eval_transform), test_idx)

trainloader_waste = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valloader_waste = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
testloader_waste = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total images: {n_total}")
print(f"Train/Val/Test: {n_train}/{n_val}/{n_test}")

Total images: 15000
Train/Val/Test: 10500/2250/2250


In [ ]:
# Validation helper for multi-class image classification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate_multiclass(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return 100.0 * correct / total

In [ ]:
# Transfer learning baseline (ResNet18) for 30 classes
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

waste_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = waste_model.fc.in_features
waste_model.fc = nn.Linear(in_features, num_classes)
waste_model = waste_model.to(device)

criterion_waste = nn.CrossEntropyLoss()
optimizer_waste = optim.Adam(waste_model.parameters(), lr=1e-4)

def train_multiclass(model, loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(loader):.4f}")

print("Training waste classifier...")
train_multiclass(waste_model, trainloader_waste, criterion_waste, optimizer_waste, epochs=5)

val_acc = evaluate_multiclass(waste_model, valloader_waste)
test_acc = evaluate_multiclass(waste_model, testloader_waste)
print(f"Validation Accuracy: {val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")

Training waste classifier...
Epoch 1/5, Loss: 1.2002
Epoch 2/5, Loss: 0.5231
Epoch 3/5, Loss: 0.3456
Epoch 4/5, Loss: 0.2520
Epoch 5/5, Loss: 0.1966
Validation Accuracy: 85.20%
Test Accuracy: 85.02%


## Kaggle Experiments: 4 Alternative Models + Overall Comparison

This block runs four alternatives and compares them against your current ResNet18 baseline.

Alternatives added:
- ResNet34 + label smoothing
- EfficientNet-B0
- DenseNet121
- ConvNeXt-Tiny

> Tip: set fewer epochs first (e.g., 3) for a quick benchmark pass.

In [ ]:
# Utilities for model benchmarking
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Fallback if this cell is run independently
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'trainloader_waste' not in globals() or 'valloader_waste' not in globals() or 'testloader_waste' not in globals():
    raise RuntimeError('Run the Kaggle DataLoader cell first (train/val/test loaders not found).')
if 'num_classes' not in globals():
    raise RuntimeError('Run the class detection cell first (num_classes not found).')


def set_backbone_trainable(model, trainable: bool):
    for p in model.parameters():
        p.requires_grad = trainable


def train_one_epoch(model, loader, criterion, optimizer, scaler, use_amp=True):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
    return running_loss / len(loader)


def evaluate_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    return 100.0 * correct / total


def run_experiment(name, model, epochs=5, lr=1e-4, weight_decay=1e-4, label_smoothing=0.0, freeze_backbone_epochs=0):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    use_amp = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # Optionally warm up by training only classifier head first.
    if freeze_backbone_epochs > 0 and hasattr(model, 'classifier'):
        set_backbone_trainable(model, False)
        for p in model.classifier.parameters():
            p.requires_grad = True

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state = copy.deepcopy(model.state_dict())
    best_val = 0.0

    start = time.time()
    for epoch in range(epochs):
        if freeze_backbone_epochs > 0 and epoch == freeze_backbone_epochs and hasattr(model, 'classifier'):
            set_backbone_trainable(model, True)
            optimizer = optim.AdamW(model.parameters(), lr=lr * 0.5, weight_decay=weight_decay)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - epoch))

        train_loss = train_one_epoch(model, trainloader_waste, criterion, optimizer, scaler, use_amp=use_amp)
        val_acc = evaluate_acc(model, valloader_waste)
        scheduler.step()

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val: {val_acc:.2f}%")

    elapsed = time.time() - start

    model.load_state_dict(best_state)
    test_acc = evaluate_acc(model, testloader_waste)

    return {
        'model': name,
        'best_val_acc': best_val,
        'test_acc': test_acc,
        'train_time_sec': elapsed,
        'best_model_state': best_state,
    }


def build_resnet34(num_classes):
    m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def build_efficientnet_b0(num_classes):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, num_classes)
    return m


def build_densenet121(num_classes):
    m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m


def build_convnext_tiny(num_classes):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = m.classifier[2].in_features
    m.classifier[2] = nn.Linear(in_features, num_classes)
    return m

In [14]:
# Alternative 1/4: ResNet34 + label smoothing
EPOCHS_BENCH = 10  # set to 3 for quick test, 8-12 for stronger final results

if 'model_results' not in globals():
    model_results = {}

model_results['ResNet34 + LS(0.1)'] = run_experiment(
    name='ResNet34 + LS(0.1)',
    model=build_resnet34(num_classes),
    epochs=EPOCHS_BENCH,
    lr=1e-4,
    weight_decay=1e-4,
    label_smoothing=0.1,
)

/tmp/ipykernel_10165/1615843433.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_10165/1615843433.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[ResNet34 + LS(0.1)] Epoch 1/10 | Loss: 1.5116 | Val: 80.62%
[ResNet34 + LS(0.1)] Epoch 2/10 | Loss: 1.0682 | Val: 82.98%
[ResNet34 + LS(0.1)] Epoch 3/10 | Loss: 0.9402 | Val: 83.82%
[ResNet34 + LS(0.1)] Epoch 4/10 | Loss: 0.8674 | Val: 85.82%
[ResNet34 + LS(0.1)] Epoch 5/10 | Loss: 0.8077 | Val: 86.00%
[ResNet34 + LS(0.1)] Epoch 6/10 | Loss: 0.7754 | Val: 86.67%
[ResNet34 + LS(0.1)] Epoch 7/10 | Loss: 0.7474 | Val: 87.38%
[ResNet34 + LS(0.1)] Epoch 8/10 | Loss: 0.7313 | Val: 87.47%
[ResNet34 + LS(0.1)] Epoch 9/10 | Loss: 0.7209 | Val: 87.29%
[ResNet34 + LS(0.1)] Epoch 10/10 | Loss: 0.7121 | Val: 87.69%


In [ ]:
# Alternative 2/4: EfficientNet-B0
if 'model_results' not in globals():
    model_results = {}

model_results['EfficientNet-B0'] = run_experiment(
    name='EfficientNet-B0',
    model=build_efficientnet_b0(num_classes),
    epochs=EPOCHS_BENCH,
    lr=2e-4,
    weight_decay=1e-4,
    label_smoothing=0.05,
)

/tmp/ipykernel_10165/1615843433.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_10165/1615843433.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[EfficientNet-B0] Epoch 1/5 | Loss: 1.5937 | Val: 80.98%
[EfficientNet-B0] Epoch 2/5 | Loss: 0.8541 | Val: 84.53%
[EfficientNet-B0] Epoch 3/5 | Loss: 0.6998 | Val: 86.40%
[EfficientNet-B0] Epoch 4/5 | Loss: 0.6008 | Val: 86.84%
[EfficientNet-B0] Epoch 5/5 | Loss: 0.5616 | Val: 87.42%


In [15]:
# Alternative 3/4: ConvNeXt-Tiny
if 'model_results' not in globals():
    model_results = {}

model_results['ConvNeXt-Tiny'] = run_experiment(
    name='ConvNeXt-Tiny',
    model=build_convnext_tiny(num_classes),
    epochs=EPOCHS_BENCH,
    lr=1e-4,
    weight_decay=5e-5,
    label_smoothing=0.1,
)

/tmp/ipykernel_10165/1615843433.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_10165/1615843433.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[ConvNeXt-Tiny] Epoch 1/10 | Loss: 1.4447 | Val: 86.09%
[ConvNeXt-Tiny] Epoch 2/10 | Loss: 0.9278 | Val: 88.93%
[ConvNeXt-Tiny] Epoch 3/10 | Loss: 0.8190 | Val: 89.33%
[ConvNeXt-Tiny] Epoch 4/10 | Loss: 0.7704 | Val: 89.11%
[ConvNeXt-Tiny] Epoch 5/10 | Loss: 0.7414 | Val: 89.29%
[ConvNeXt-Tiny] Epoch 6/10 | Loss: 0.7279 | Val: 89.69%
[ConvNeXt-Tiny] Epoch 7/10 | Loss: 0.7144 | Val: 89.33%
[ConvNeXt-Tiny] Epoch 8/10 | Loss: 0.7071 | Val: 89.47%
[ConvNeXt-Tiny] Epoch 9/10 | Loss: 0.6989 | Val: 89.38%
[ConvNeXt-Tiny] Epoch 10/10 | Loss: 0.6969 | Val: 89.38%


In [13]:
# Final comparison (excluding DenseNet): ResNet18 baseline + selected alternatives
import pandas as pd

if 'model_results' not in globals() or len(model_results) == 0:
    raise RuntimeError('No alternative model results found. Run the alternative cells first.')

# Exclude DenseNet from final summary by request
results = [v for k, v in model_results.items() if k != 'DenseNet121']

if 'val_acc' in globals() and 'test_acc' in globals():
    results.append({
        'model': 'ResNet18 baseline',
        'best_val_acc': float(val_acc),
        'test_acc': float(test_acc),
        'train_time_sec': float('nan'),
        'best_model_state': None,
    })
else:
    print('Note: ResNet18 baseline metrics not found; showing alternatives only.')

summary = pd.DataFrame(results)
summary_view = summary.drop(columns=['best_model_state'], errors='ignore').sort_values(
    by=['best_val_acc', 'test_acc'], ascending=False
).reset_index(drop=True)

best_model_name = summary_view.iloc[0]['model']
best_state = summary.sort_values(by=['best_val_acc', 'test_acc'], ascending=False).iloc[0].get('best_model_state', None)

print(f'Best model (excluding DenseNet): {best_model_name}')
display(summary_view)

Best model (excluding DenseNet): ConvNeXt-Tiny


,model,best_val_acc,test_acc,train_time_sec
0,ConvNeXt-Tiny,90.088889,88.800000,803.295147
1,ResNet34 + LS(0.1),88.000000,86.844444,406.968678
2,EfficientNet-B0,87.422222,87.555556,412.009716
3,ResNet18 baseline,85.200000,85.022222,NaN


In [12]:
# Alternative 4/4 (run last): DenseNet121
if 'model_results' not in globals():
    model_results = {}

model_results['DenseNet121'] = run_experiment(
    name='DenseNet121',
    model=build_densenet121(num_classes),
    epochs=EPOCHS_BENCH,
    lr=8e-5,
    weight_decay=1e-4,
    label_smoothing=0.02,
    freeze_backbone_epochs=2,
)

/tmp/ipykernel_10165/1615843433.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_10165/1615843433.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[DenseNet121] Epoch 1/5 | Loss: 3.1174 | Val: 38.58%


KeyboardInterrupt: 

### Save checkpoint for `inference_app.py`

Run the next cell **after** training the model you want to deploy. It saves both the `.pth` file and **`deployment_config.json`** so they stay in sync.

**Architecture strings** (must match `inference_app.py`):

| Model you trained | Set `ARCH` to |
|-------------------|----------------|
| ResNet18 baseline (`waste_model`) | `resnet18` |
| ResNet34 alternative | `resnet34` |
| EfficientNet-B0 | `efficientnet_b0` |
| DenseNet121 | `densenet121` |
| ConvNeXt-Tiny | `convnext_tiny` |

If you deploy a different model, change the `torch.save(...)` line to use that model variable and set `ARCH` accordingly.

In [17]:
# Save weights + deployment_config.json (run from cnn/ after training the model you deploy)
# Set ARCH to match the variable you trained — must match inference_app.py builders:
#   resnet18  -> waste_model (baseline)
#   resnet34  -> model from ResNet34 run_experiment
#   efficientnet_b0, densenet121, convnext_tiny -> same names
import json
import torch
from pathlib import Path

ROOT = Path.cwd()
CHECKPOINT_NAME = "best_waste_model.pth"
ARCH = "resnet18"  # change if you deploy ResNet34, EfficientNet, etc.

out = ROOT / CHECKPOINT_NAME
torch.save(waste_model.state_dict(), out)

cfg_path = ROOT / "deployment_config.json"
cfg = {"checkpoint": CHECKPOINT_NAME, "architecture": ARCH}
cfg_path.write_text(json.dumps(cfg, indent=2) + "\n", encoding="utf-8")

print("Saved weights:", out.resolve())
print("Wrote config:", cfg_path.resolve())
print("ARCH used:", ARCH, "- must match how this model was built in the notebook.")

Saved weights: /home/teoyongsong/cnn/best_waste_model.pth
Wrote config: /home/teoyongsong/cnn/deployment_config.json
ARCH used: resnet18 - must match how this model was built in the notebook.
